# Функции диагностики
Подключаются из 06.

In [ ]:
def select_dev(manifest, limit=5000, seed=42):
    frame = manifest[manifest.split == "dev"].sort_values("crop_id").copy()
    if frame.empty:
        raise ValueError("В manifest нет dev")
    return (
        frame.sample(min(limit, len(frame)), random_state=seed)
        .sort_values("crop_id")
        .reset_index(drop=True)
    )


def geometry(width, height, target_h=48, target_w=192):
    if width <= 0 or height <= 0:
        raise ValueError("Нулевой размер изображения")
    scale = min(target_w / width, target_h / height)
    w = max(1, min(target_w, round(width * scale)))
    h = max(1, min(target_h, round(height * scale)))
    return dict(
        width=width,
        height=height,
        aspect=width / height,
        effective_height=h,
        effective_width=w,
        occupied_fraction=w * h / (target_w * target_h),
    )


def image_stats(frame, target_h=48, target_w=192):
    rows = []
    for row in tqdm(frame.itertuples(), total=len(frame), desc="Размеры и декодирование"):
        try:
            im = open_rgb(ROOT / row.path)
            rows.append(dict(crop_id=row.crop_id, **geometry(*im.size, target_h, target_w)))
        except Exception as exc:
            raise RuntimeError(f"Не удалось прочитать {row.path}: {exc}") from exc
    return pd.DataFrame(rows)


def load_checkpoint(path, device):
    state = torch.load(path, map_location="cpu", weights_only=True)
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(state["model"])
    model.eval()
    return model, state
